[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/26_lora.ipynb)

# 🟠 Medium: LoRA (Low-Rank Adaptation)

Implement **LoRA** — parameter-efficient fine-tuning for large models.

$$h = W_0 x + \frac{\alpha}{r} B A x$$

### Signature
```python
class LoRALinear(nn.Module):
    def __init__(self, in_features, out_features, rank, alpha=1.0): ...
    def forward(self, x: Tensor) -> Tensor: ...
```

### Requirements
- `self.linear`: frozen `nn.Linear` (weight & bias `requires_grad=False`)
- `self.lora_A`: `nn.Parameter(rank, in_features)` — random init
- `self.lora_B`: `nn.Parameter(out_features, rank)` — **zero** init
- Scaling: `alpha / rank`

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 4.1 MB/s eta 0:00:00


In [2]:
import torch
import torch.nn as nn

In [3]:
# ✏️ YOUR IMPLEMENTATION HERE

class LoRALinear(nn.Module):
    def __init__(self, in_features, out_features, rank, alpha=1.0):
        super().__init__()
        self.linear=nn.Linear(in_features,out_features)
        for param in self.linear.parameters():
          param.requires_grad=False

        self.lora_A=nn.Parameter(torch.empty(rank,in_features))
        self.lora_B=nn.Parameter(torch.empty(out_features,rank))

        #init
        nn.init.kaiming_uniform_(self.lora_A)
        nn.init.zeros_(self.lora_B)

        self.rank=rank
        self.alpha=alpha
        self.scaling = alpha/rank



    def forward(self, x):
        base_output=self.linear(x)

        Ax=torch.matmul(x,self.lora_A.transpose(-1,-2))
        BAx=torch.matmul(Ax,self.lora_B.transpose(-1,-2))

        return base_output+self.scaling*BAx


In [4]:
# 🧪 Debug
layer = LoRALinear(16, 8, rank=4)
x = torch.randn(2, 16)
print('Output:', layer(x).shape)
print('Trainable:', sum(p.numel() for p in layer.parameters() if p.requires_grad))
print('Total:    ', sum(p.numel() for p in layer.parameters()))

Output: torch.Size([2, 8])
Trainable: 96
Total:     232


In [5]:
# ✅ SUBMIT
from torch_judge import check
check('lora')


🧪 Testing: LoRA (Low-Rank Adaptation) (Medium)
──────────────────────────────────────────────────
  ✅ [1/5] Base weights frozen (1.0ms)
  ✅ [2/5] LoRA parameter shapes (0.6ms)
  ✅ [3/5] B=0 means output equals base (42.2ms)
  ✅ [4/5] Only LoRA params get gradients (29.2ms)
  ✅ [5/5] Forward computation (2.7ms)
──────────────────────────────────────────────────
  🎉 All 5 tests passed! (75.7ms total)
  Progress saved. Run status() to see your dashboard.

